# Step 3: Generate Dataset with Qwen3-32B via vLLM

For each FAERS record, Qwen3-32B fills in only the fields that cannot be derived from structured data:

| Field | Source | Reliability |
|-------|--------|-------------|
| `narrative` | **Qwen3-32B generated** (all records) | Good — consistent with ground truth labels |
| `seriousness`, `criteria` | FAERS (ground truth) | ✅ High |
| `meddra_pt`, `meddra_soc` | MedDRA lookup (notebook 02) | ✅ High |
| `who_umc_category` | FAERS fields — deterministic (notebook 01) | ✅ Medium |
| `labelling_status` | **FDA drug label API + Qwen3-32B** | ✅ Medium-High |
| ~~`naranjo_score`~~ | Dropped — unreliable from synthetic narrative | — |

## Reliability improvements vs. original design
- **WHO-UMC**: computed from FAERS `drugadditional`/`drugrecurrence`/`outcome` fields — no LLM
- **Labelling status**: Qwen3-32B receives the actual FDA prescribing information adverse reactions
  section as context, rather than relying on parametric memory alone

## Prerequisites
```bash
vllm serve Qwen/Qwen3-32B --dtype bfloat16 --max-model-len 4096 --port 8000
```

## Checkpointing
Results are saved every 200 records. Re-running the notebook resumes from the last checkpoint.

In [ ]:
%pip install openai httpx tqdm -q

In [ ]:
import json
import asyncio
import time
import re
from pathlib import Path
from tqdm.notebook import tqdm
from openai import AsyncOpenAI
import httpx

DATA_DIR = Path("data")

FAERS_FILE        = DATA_DIR / "faers_raw.json"
LOOKUP_FILE       = DATA_DIR / "meddra_lookup.json"
LABEL_CACHE_FILE  = DATA_DIR / "drug_labels.json"   # cached FDA label ADR sections
OUTPUT_FILE       = DATA_DIR / "dataset_raw.json"

VLLM_BASE_URL = "http://localhost:8000/v1"
MODEL_NAME    = "Qwen/Qwen3-32B"
FDA_LABEL_URL = "https://api.fda.gov/drug/label.json"

CONCURRENCY      = 8
CHECKPOINT_EVERY = 200

print("Config loaded.")

In [ ]:
# ── Load input data ──────────────────────────────────────────────────────────
with open(FAERS_FILE) as f:
    faers_records = json.load(f)

with open(LOOKUP_FILE) as f:
    pt_lookup = json.load(f)

print(f"FAERS records: {len(faers_records)}")
print(f"MedDRA lookup entries: {len(pt_lookup)}")

In [ ]:
# ── vLLM health check ────────────────────────────────────────────────────────
import httpx

try:
    resp = httpx.get(f"{VLLM_BASE_URL}/models", timeout=5)
    models = resp.json()
    print(f"✓ vLLM is running. Available models:")
    for m in models.get("data", []):
        print(f"  - {m['id']}")
except Exception as e:
    print(f"✗ vLLM not reachable: {e}")
    print("Start vLLM first:")
    print("  vllm serve Qwen/Qwen3-32B --dtype bfloat16 --max-model-len 4096 --port 8000")

In [ ]:
# ── MedDRA fuzzy lookup (same as notebook 02) ────────────────────────────────
from rapidfuzz import process, fuzz

ALL_PT_KEYS = list(pt_lookup.keys())

def lookup_pt(pt_name: str, threshold: int = 80) -> dict:
    """Return MedDRA hierarchy dict for a PT name. Returns empty dict if not found."""
    key = pt_name.lower().strip()
    if key in pt_lookup:
        return pt_lookup[key]
    result = process.extractOne(key, ALL_PT_KEYS, scorer=fuzz.WRatio, score_cutoff=threshold)
    if result:
        return pt_lookup[result[0]]
    return {}

print("lookup_pt ready.")

In [ ]:
# ── Fetch FDA drug labels (adverse_reactions section) ────────────────────────
# One-time fetch for all unique drugs. Cached to drug_labels.json.
# Gives Qwen3 actual FDA label text as context for labelling assessment.

async def _fetch_label(client: httpx.AsyncClient, drug_name: str) -> str:
    searches = [
        f'openfda.brand_name:"{drug_name}"',
        f'openfda.generic_name:"{drug_name}"',
        f'openfda.substance_name:"{drug_name}"',
    ]
    for search in searches:
        try:
            resp = await client.get(
                FDA_LABEL_URL,
                params={"search": search, "limit": 1},
                timeout=15.0,
            )
            if resp.status_code == 200:
                results = resp.json().get("results", [])
                if results:
                    label = results[0]
                    adr = label.get("adverse_reactions", [])
                    if adr:
                        return adr[0][:2500]
                    warn = label.get("warnings_and_cautions", label.get("warnings", []))
                    if warn:
                        return warn[0][:1500]
            elif resp.status_code == 404:
                continue
        except Exception:
            pass
    return ""


async def fetch_all_labels(drugs: list, concurrency: int = 5) -> dict:
    sem = asyncio.Semaphore(concurrency)

    async def bounded(drug):
        async with sem:
            label = await _fetch_label(client, drug)
            await asyncio.sleep(0.15)
            return drug, label

    labels = {}
    async with httpx.AsyncClient() as client:
        tasks = [bounded(d) for d in drugs]
        with tqdm(total=len(drugs), desc="Fetching FDA labels") as pbar:
            for coro in asyncio.as_completed(tasks):
                drug, label = await coro
                labels[drug] = label
                pbar.update(1)
                pbar.set_postfix({"found": sum(1 for v in labels.values() if v)})
    return labels


if LABEL_CACHE_FILE.exists():
    with open(LABEL_CACHE_FILE) as f:
        drug_labels = json.load(f)
    found = sum(1 for v in drug_labels.values() if v)
    print(f"Loaded {len(drug_labels)} cached drug labels ({found} with text)")
else:
    unique_drugs = list(set(r["drug_name"] for r in faers_records))
    print(f"Fetching FDA labels for {len(unique_drugs)} unique drugs...")
    drug_labels = await fetch_all_labels(unique_drugs)
    with open(LABEL_CACHE_FILE, "w") as f:
        json.dump(drug_labels, f, indent=2)
    found = sum(1 for v in drug_labels.values() if v)
    print(f"Done. Found labels for {found}/{len(unique_drugs)} drugs ({found/len(unique_drugs):.1%})")
    print(f"Saved to {LABEL_CACHE_FILE}")

In [ ]:
# ── Prompt templates ─────────────────────────────────────────────────────────

# Action taken mapping — used in narrative prompt so the text is consistent
# with the WHO-UMC dechallenge signal already computed from FAERS fields
ACTION_TAKEN = {
    "1": "The drug was discontinued.",
    "2": "The drug dose was reduced.",
    "3": "The drug was continued without change.",
    "4": "",   # unknown — omit from narrative
    "":  "",   # not reported — omit
}

# One-shot example anchors clinical narrative style:
# third person, temporal onset, action taken, outcome — ICH E2D format
_NARRATIVE_EXAMPLE = """\
### Example
Input:
Drug: Atorvastatin 40 mg once daily | Adverse event: Myopathy | \
Indication: Hypercholesterolaemia | Patient: 58-year-old male | \
Outcome: Recovered/Resolved | Serious: Non-serious | Action: Drug withdrawn

Narrative:
A 58-year-old male patient with hypercholesterolaemia was prescribed atorvastatin \
40 mg once daily. After approximately three months of treatment, he developed \
bilateral proximal muscle weakness and myalgia. Creatine kinase levels were \
elevated on laboratory testing. Atorvastatin was discontinued and his symptoms \
resolved gradually over four weeks.\
"""

NARRATIVE_PROMPT = """You are a pharmacovigilance medical writer. \
Write a concise clinical safety narrative (3-5 sentences) in ICH E2D format.

{example}

### Now write a narrative for:
Input:
Drug: {drug_name}{dose_info} | Adverse event: {reaction_pt} | \
Indication: {indication} | Patient: {age_sex} | \
Outcome: {outcome} | Serious: {serious} | Action: {action_taken}

Rules:
- Third person, past tense
- Mention temporal relationship (e.g. "after X weeks of treatment")
- Include the action taken with respect to the drug
- State the outcome
- Return ONLY the narrative text, no labels or JSON

Narrative:"""


# Primary path — FDA label text available
LABELLING_PROMPT_WITH_LABEL = """You are a senior pharmacovigilance specialist assessing labelling status.

Drug: {drug_name}
Adverse Event (MedDRA PT): {reaction_pt}

Adverse Reactions section from the official FDA prescribing information:
---
{label_excerpt}
---

Is "{reaction_pt}" an Expected or Unexpected adverse reaction based on this label?
- Expected: this reaction (or a closely related term) is mentioned in the adverse reactions section
- Unexpected: this reaction is not listed in the label

Return ONLY valid JSON with exactly these two keys:
{{
  "labelling_status": "Expected" or "Unexpected",
  "labelling_evidence": "<one sentence citing specific label text or explaining why it is not listed>"
}}"""


# Fallback — no FDA label found for this drug
LABELLING_PROMPT_NO_LABEL = """You are a senior pharmacovigilance specialist assessing labelling status.

Drug: {drug_name}
Adverse Event (MedDRA PT): {reaction_pt}
Indication: {indication}

Based on your medical knowledge of {drug_name} and its drug class, is "{reaction_pt}" \
a known adverse reaction typically listed in the prescribing information?
- Expected: the reaction is commonly associated with this drug or its class
- Unexpected: the reaction is not typically associated with this drug or its class

Return ONLY valid JSON with exactly these two keys:
{{
  "labelling_status": "Expected" or "Unexpected",
  "labelling_evidence": "<one sentence explaining the pharmacological basis>"
}}"""


def extract_json(text: str) -> dict | None:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"```(?:json)?\s*({.*?})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r"({.*})", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    return None


print("Prompts defined.")

In [ ]:
# ── Async generation ──────────────────────────────────────────────────────────

client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")


async def generate_text(prompt: str, max_tokens: int = 300, temperature: float = 0.4) -> str:
    response = await client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return response.choices[0].message.content.strip()


async def process_record(record: dict) -> dict | None:
    """
    Produces one training example from a FAERS record.
    LLM calls: (1) narrative generation  (2) labelling assessment.
    WHO-UMC: read from FAERS field — no LLM needed.
    Naranjo: dropped.
    """
    try:
        drug     = record["drug_name"]
        reaction = record["reaction_pt"]

        # ── MedDRA lookup ──────────────────────────────────────────────────────
        meddra = lookup_pt(reaction)
        if not meddra:
            return None

        # ── Narrative generation ───────────────────────────────────────────────
        age        = record.get("patient_age", "")
        sex        = record.get("patient_sex", "unknown")
        dose       = record.get("drug_dose", "")
        indication = record.get("drug_indication", "") or "unknown indication"
        outcome    = record.get("reaction_outcome", "Unknown")
        serious    = "Serious" if record.get("serious") else "Non-serious"
        action     = ACTION_TAKEN.get(record.get("drug_additional", ""), "")

        narrative_prompt = NARRATIVE_PROMPT.format(
            example     = _NARRATIVE_EXAMPLE,
            drug_name   = drug,
            dose_info   = f" {dose}" if dose else "",
            reaction_pt = reaction,
            indication  = indication,
            age_sex     = f"{age}-year-old {sex}" if age else sex,
            outcome     = outcome,
            serious     = serious,
            action_taken = action if action else "Not reported",
        )
        narrative = await generate_text(narrative_prompt, max_tokens=250, temperature=0.5)
        # Strip any leading "Narrative:" label the model might echo back
        narrative = re.sub(r"^Narrative:\s*", "", narrative, flags=re.IGNORECASE).strip()

        # ── Labelling assessment (FDA label context when available) ────────────
        label_text = drug_labels.get(drug, "")
        if label_text:
            labelling_prompt = LABELLING_PROMPT_WITH_LABEL.format(
                drug_name     = drug,
                reaction_pt   = reaction,
                label_excerpt = label_text,
            )
        else:
            labelling_prompt = LABELLING_PROMPT_NO_LABEL.format(
                drug_name   = drug,
                reaction_pt = reaction,
                indication  = indication,
            )

        labelling_text = await generate_text(labelling_prompt, max_tokens=200, temperature=0.1)
        assessment = extract_json(labelling_text)

        if not assessment or assessment.get("labelling_status") not in ("Expected", "Unexpected"):
            return None

        # ── Assemble ───────────────────────────────────────────────────────────
        return {
            "id":               record["id"],
            "narrative":        narrative,
            "narrative_source": "generated",
            "seriousness":           serious,
            "seriousness_criteria":  record.get("seriousness_criteria", []),
            "meddra_llt":       meddra.get("meddra_llt",      meddra["meddra_pt"]),
            "meddra_llt_code":  meddra.get("meddra_llt_code"),
            "meddra_pt":        meddra["meddra_pt"],
            "meddra_pt_code":   meddra.get("meddra_pt_code"),
            "meddra_soc":       meddra.get("meddra_soc", ""),
            "meddra_soc_code":  meddra.get("meddra_soc_code"),
            "labelling_status":   assessment.get("labelling_status"),
            "labelling_evidence": assessment.get("labelling_evidence", ""),
            "label_source":       "fda_label" if label_text else "llm_knowledge",
            "who_umc_category":   record.get("who_umc_preliminary", "Unassessable"),
        }

    except Exception:
        return None


print("generate_text and process_record defined.")

In [ ]:
# ── Quick single-record test before full run ─────────────────────────────────
test_record = faers_records[0]
print(f"Testing with record: drug={test_record['drug_name']}, reaction={test_record['reaction_pt']}")

result = await process_record(test_record)

if result:
    print("\n✓ Test passed. Sample output:")
    print(json.dumps(result, indent=2))
else:
    print("✗ Test failed. Check vLLM server and FAERS data.")

In [ ]:
# ── Main async generation loop ────────────────────────────────────────────────

async def run_generation(records: list, concurrency: int = CONCURRENCY) -> list:
    sem     = asyncio.Semaphore(concurrency)
    results = list(completed)   # seed with already-completed records
    failed  = 0

    async def bounded(record):
        async with sem:
            return await process_record(record)

    tasks = [bounded(r) for r in records]

    with tqdm(total=len(records), desc="Generating", unit="rec") as pbar:
        for i, coro in enumerate(asyncio.as_completed(tasks)):
            result = await coro
            if result:
                results.append(result)
            else:
                failed += 1

            pbar.update(1)
            pbar.set_postfix({"done": len(results), "failed": failed})

            if (i + 1) % CHECKPOINT_EVERY == 0:
                with open(OUTPUT_FILE, "w") as f:
                    json.dump(results, f)
                pbar.write(f"  Checkpoint: {len(results)} records saved")

    return results


t0          = time.time()
all_results = await run_generation(todo)
elapsed     = time.time() - t0

with open(OUTPUT_FILE, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nDone in {elapsed/60:.1f} min")
print(f"Total records: {len(all_results)}")
print(f"Saved to {OUTPUT_FILE}")

In [ ]:
# ── Quality check ─────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results)
print(f"Total records: {len(df)}")

print(f"\nSeriousness distribution:")
print(df["seriousness"].value_counts())

print(f"\nLabelling status:")
print(df["labelling_status"].value_counts())

print(f"\nLabel source (fda_label vs llm_knowledge):")
print(df["label_source"].value_counts())
print(f"  → {(df['label_source']=='fda_label').mean():.1%} of records used actual FDA label text")

print(f"\nWHO-UMC category (from FAERS — deterministic):")
print(df["who_umc_category"].value_counts())

print(f"\nTop 10 SOCs:")
print(df["meddra_soc"].value_counts().head(10))

print(f"\nNarrative length (chars):")
print(df["narrative"].str.len().describe().round(0))

In [ ]:
# ── Quality check ─────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results)
print(f"Total records: {len(df)}")
print(f"\nNarrative source:")
print(df["narrative_source"].value_counts())
print(f"\nSeriousness distribution:")
print(df["seriousness"].value_counts())
print(f"\nLabelling status:")
print(df["labelling_status"].value_counts())
print(f"\nNaranjo category:")
print(df["naranjo_category"].value_counts())
print(f"\nWHO-UMC category:")
print(df["who_umc_category"].value_counts())
print(f"\nNaranjo score distribution:")
print(df["naranjo_score"].describe())

# Check SOC coverage
print(f"\nTop 10 SOCs:")
print(df["meddra_soc"].value_counts().head(10))